<a href="https://colab.research.google.com/github/rajeshabraham/demopub/blob/main/Qwen3_4B_finetune_salesbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [2]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 1024 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-Base",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.2.1: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.32G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/166 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/617 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [14]:
from transformers import TextStreamer
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

tokenizer.chat_template = """{% for message in messages %}
<|{{ message.role }}|>
{{ message.content }}
{% endfor %}
{% if add_generation_prompt %}<|assistant|>
{% endif %}"""

SYSTEM_PROMPT = (
    "You are a B2C Mobile Postpaid Sales Expert for chat.\n"
    "Use ONLY the JSON fields provided in the input.\n"
    "Format your response as: Greeting → Needs Check → Recommendation (one) → "
    "Alternative (one) → Next Steps → Disclaimers.\n"
    "If OOD/unsafe, politely refuse and redirect.\n"
)

user_json = {
  "PRODUCT_CATALOG": {
    "mobile_postpaid": [
      {"id":"P1","name":"Unlimited Lite","data_gb":"unlimited","hotspot_gb":5,"price":50},
      {"id":"P2","name":"Unlimited Plus","data_gb":"unlimited","hotspot_gb":30,"price":75}
    ]
  },
  "ACTIVE_PROMOTIONS":[
    {"applies_to":"Unlimited Plus","type":"bill_credit","value":10,"term_months":12,"desc":"$10/mo credit for 12 months"}
  ],
  "CUSTOMER_PROFILE":{"lines":1,"avg_monthly_data_gb":85,"hotspot_need_gb":10,"priority":"speed"},
  "POLICY_SNIPPETS":"Taxes/fees extra; credit check may apply; AutoPay required for some discounts."
}
question = "My bill spiked but I still need fast 5G and hotspot. What can you do?"

messages = [
  {"role": "system", "content": SYSTEM_PROMPT},
  {"role": "user", "content": f"JSON:\n{user_json}\n\nQuestion:\n{question}"}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")

In [37]:
prompt.shape

torch.Size([1, 309])

In [19]:
streamer = TextStreamer(tokenizer, skip_prompt=True)
_ = model.generate(
    input_ids = prompt,
    max_new_tokens=512,
    do_sample=False,      # or True with temperature for more variety
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.05,
    eos_token_id=tokenizer.eos_token_id,
    streamer = streamer
)

Hey there! I see **1 line** and about **85GB** monthly use. You also need around **10GB** hotspot. I recommend **Unlimited Plus** (≈ **$75**/mo) with **unlimited data** and **30GB** hotspot. **Promo:** $10/mo credit for 12 months. Or, if you prefer, **Unlimited Lite** (≈ **$50**/mo) with **unlimited data** and **5GB** hotspot for a lower monthly cost. Would you like me to lock this in or compare one more option? (*Taxes/fees extra; credit check may apply; AutoPay required for some discounts.*)
<|endoftext|>


In [5]:
import json

src = "/content/sft_telco_sales_en.txt"
out = "/content/sft_telco_sales_en_chat.jsonl"

SYSTEM_PROMPT = (
    "You are a B2C Mobile Postpaid Sales Expert for chat.\n"
    "Use ONLY the JSON fields provided in the input.\n"
    "Format your response as: Greeting → Needs Check → Recommendation → "
    "Alternative → Next Steps → Disclaimers.\n"
    "If OOD/unsafe, politely refuse and redirect.\n"
)

with open(src, "r", encoding="utf-8") as fin, \
     open(out, "w", encoding="utf-8") as fout:

    for line in fin:
        line = line.strip()
        if not line:
            continue

        try:
            item = json.loads(line)
        except:
            # Skip malformed lines
            continue

        instruction = item.get("instruction", "").strip()
        input_data = item.get("input", {})
        output = item.get("output", "").strip()

        # Build message list
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": f"Instruction:\n{instruction}\n\nInput:\n{json.dumps(input_data, indent=2)}"
            },
            {"role": "assistant", "content": output}
        ]

        fout.write(json.dumps({"messages": messages}, ensure_ascii=False) + "\n")

print("DONE →", out)


DONE → /content/sft_telco_sales_en_chat.jsonl


In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 8, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],

                      # "embed_tokens", "lm_head",], # Add for continual pretraining
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = True,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

# model.load_adapter("path/to/your/unzipped/folder")

Unsloth 2026.2.1 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [17]:
from datasets import load_dataset
from unsloth import UnslothTrainer, UnslothTrainingArguments

data = load_dataset(
    "json",
    data_files="/content/sft_telco_sales_en_chat.jsonl",
    split="train"
)

def formatting_func(examples):
    # 'examples' is a dictionary of lists (e.g., {"messages": [[...], [...]]})
    instructions = examples["messages"]
    eos_token = tokenizer.eos_token
    texts = []

    for messages in instructions:
        text = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=False,
            tokenize=False,
        )
        texts.append(text + eos_token)

    return texts # Returns a flat list of strings: ["text1", "text2", ...]


trainer = UnslothTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = data,
    # dataset_text_field = "text",
    formatting_func = formatting_func,
    max_seq_length = max_seq_length,
    dataset_num_proc = 4,
    dataset_batched=True,
    args = UnslothTrainingArguments(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        warmup_ratio = 0.03,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none"
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=5):   0%|          | 0/500 [00:00<?, ? examples/s]

In [11]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
5.506 GB of memory reserved.


In [18]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 3 | Total steps = 96
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 16,515,072 of 4,038,983,168 (0.41% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,0.062800
20,0.047900
30,0.045500


KeyboardInterrupt: 

In [20]:
# This saves the LoRA adapters and the tokenizer
model.save_pretrained("valentina_sft_lora_v2")
tokenizer.save_pretrained("valentina_sft_lora_v2")

('valentina_sft_lora_v2/tokenizer_config.json',
 'valentina_sft_lora_v2/special_tokens_map.json',
 'valentina_sft_lora_v2/chat_template.jinja',
 'valentina_sft_lora_v2/vocab.json',
 'valentina_sft_lora_v2/merges.txt',
 'valentina_sft_lora_v2/added_tokens.json',
 'valentina_sft_lora_v2/tokenizer.json')

In [21]:
import shutil
from google.colab import files

# Zip the folder
shutil.make_archive("valentina_sft_lora_v2", 'zip', "valentina_sft_lora_v2")

# Download to your computer
files.download("valentina_sft_lora_v2.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import json
from datasets import Dataset

# 1. Define your constants
SYSTEM_PROMPT = (
    "You are a B2C Mobile Postpaid Sales Expert for chat.\n"
    "Use ONLY the JSON fields provided in the input.\n"
    "Format your response as: Greeting → Needs Check → Recommendation (one) → "
    "Alternative (one) → Next Steps → Disclaimers.\n"
    "If OOD/unsafe, politely refuse and redirect.\n"
)

def prepare_grpo_data(file_path, tokenizer):
    prepared_data = []
    eos_token = tokenizer.eos_token

    with open(file_path, 'r') as f:
        for line in f:
            data = json.loads(line)

            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"Prompt: {data['prompt']}\n\nContext: {json.dumps(data['context_json'])}"}
            ]
            formatted_prompt = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )

            # Pair 1: Best (0) vs. Medium (1) -> Teaches Completeness
            dpo_dataset.append({
                "prompt": formatted_prompt,
                "chosen": data['completions'][0] + eos_token,
                "rejected": data['completions'][1] + eos_token
            })

            # Pair 2: Best (0) vs. Worst (2) -> Teaches Format/Compliance
            dpo_dataset.append({
                "prompt": formatted_prompt,
                "chosen": data['completions'][0] + eos_token,
                "rejected": data['completions'][2] + eos_token
            })

    return Dataset.from_list(dpo_dataset)

# 2. Load and transform
# Replace 'grpo_telco_sales_en.txt' with your actual path
grpo_dataset = prepare_grpo_data("grpo_telco_sales_en.txt", tokenizer)

In [ ]:
from unsloth import PatchDPOTrainer
from trl import DPOConfig
# Note: Unsloth often still imports via 'PatchDPOTrainer' name for
# compatibility, but it uses the optimized Unsloth kernels.

dpo_trainer = PatchDPOTrainer(
    model = model,
    ref_model = None, # Unsloth handles the reference model internally!
    args = DPOConfig(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        warmup_ratio = 0.1,
        num_train_epochs = 1, # DPO is powerful; 1 epoch is usually enough
        learning_rate = 5e-7, # DPO likes a very small LR
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        optim = "adamw_8bit",
    ),
    train_dataset = grpo_dataset, # The multi-pair dataset we built
    tokenizer = tokenizer,
)

dpo_trainer.train()

Next, we shall save the model to GGUF / llama.cpp

We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

We also support saving to multiple GGUF options in a list fashion! This can speed things up by 10 minutes or more if you want multiple export formats!

In [ ]:
# Save to 8bit Q8_0
if True: model.save_pretrained_gguf("llama_finetune", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("HF_USERNAME/llama_finetune", tokenizer, token = "YOUR_HF_TOKEN")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("llama_finetune", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("HF_USERNAME/llama_finetune", tokenizer, quantization_method = "f16", token = "YOUR_HF_TOKEN")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("llama_finetune", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("HF_USERNAME/llama_finetune", tokenizer, quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/llama_finetune", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "YOUR_HF_TOKEN", # Get a token at https://huggingface.co/settings/tokens
    )

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.
Unsloth: You have 1 CPUs. Using `safe_serialization` is 10x slower.
We shall switch to Pytorch saving, which will take 3 minutes and not 30 minutes.
To force `safe_serialization`, set it to `None` instead.
Unsloth: Kaggle/Colab has limited disk space. We need to delete the downloaded
model which will save 4-16GB of disk space, allowing you to save on Kaggle/Colab.
Unsloth: Will remove a cached repo with size 5.7G

Unsloth: Merging 4bit and LoRA weights to 16bit...
Unsloth: Will use up to 5.49 out of 12.67 RAM for saving.

100%|██████████| 32/32 [01:41<00:00,  3.18s/it]

Unsloth: Saving tokenizer... Done.
Unsloth: Saving model... This might take 5 minutes for Llama-7b...
Unsloth: Saving model/pytorch_model-00001-of-00004.bin...
Unsloth: Saving model/pytorch_model-00002-of-00004.bin...
Unsloth: Saving model/pytorch_model-00003-of-00004.bin...
Unsloth: Saving model/pytorch_model-00004-of-00004.bin...
Done.

Unsloth: Converting llama model. Can use fast conversion = False.

==((====))==  Unsloth: Conversion from QLoRA to GGUF information
   \\   /|    [0] Installing llama.cpp will take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF 16bits will take 3 minutes.
\        /    [2] Converting GGUF 16bits to ['q8_0'] will take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: [0] Installing llama.cpp. This will take 3 minutes...
Unsloth: [1] Converting model at model into q8_0 GGUF format.
The output location will be ./model/unsloth.Q8_0.gguf
This will take 3 minutes...
INFO:hf-to-gguf:Loading model: model
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:gguf: loading model weight map from 'pytorch_model.bin.index.json'
INFO:hf-to-gguf:gguf: loading model part 'pytorch_model-00001-of-00004.bin'
INFO:hf-to-gguf:token_embd.weight,           torch.float16 --> Q8_0, shape = {4096, 128256}
INFO:hf-to-gguf:blk.0.attn_q.weight,         torch.float16 -

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### We removed it in GGUF's chat template for you.

Unsloth: Conversion completed! Output location: ./model/unsloth.Q8_0.gguf
Unsloth: Saved Ollama Modelfile to model/Modelfile